# Chapter 11 · Validating data — Lab 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab11_validating.ipynb)

**Three questions, three instruments, on real data.** Starting from the slice Lab 10 produced (or a fresh FineWeb-Edu sample), you will ask the chapter's three questions and get numbers: *does it match the target?* — a BETR-lite ranking against benchmark training items, with the retained fraction set by your compute budget; *is it good?* — a judge-calibration harness with the sample-size formula and a labeling sheet for a human audit; *does the model already know it?* — Min-K% Prob on documents, IFD on UltraChat-200k pairs, and pass-rate bins on GSM8K under a small base model. The last section runs the completion test for contamination on GSM8K with fresh controls.

CPU is enough for everything except the pass-rate section, which is ~20× faster on a GPU runtime. Budget ~15 minutes on CPU with the default sample sizes.

*Omar U. Florez, PhD · Efficient Training of LLMs · Lab 11*

In [ ]:
%pip -q install datasets sentence-transformers transformers torch scikit-learn rouge-score pyarrow
import re, json, time, collections, itertools, pathlib
import numpy as np, matplotlib.pyplot as plt
from datasets import load_dataset

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
N_DOCS = 5_000
AUDIT = {}      # the validation record: relevance, quality, membership — one entry per gate

## 0 · The candidate set

Lab 10 wrote `fineweb_edu_lab10/data.parquet`. If it is in the working directory we use it; otherwise we stream a fresh 5k-document sample so this lab runs standalone.

In [ ]:
p = pathlib.Path("fineweb_edu_lab10/data.parquet")
if p.exists():
    import pyarrow.parquet as pq
    docs = pq.read_table(p).to_pylist()[:N_DOCS]; print("loaded Lab 10 output:", len(docs))
else:
    stream = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    docs = list(itertools.islice(stream, N_DOCS)); print("streamed fresh sample:", len(docs))
texts = [d["text"] for d in docs]

## 1 · Does it match the target? BETR-lite

Chapter 11.2. Embed the **training** items of a broad suite (here: MMLU auxiliary-train, GSM8K train, ARC-Challenge train — a few hundred each), embed the documents, let every item rank every document by cosine similarity, and give each document its **best rank**: $s(x) = \max_j \mathrm{rank}_j(x)$. Positives are the top 10%. Then read what the ranking chose, and what it rejected — that is the whole method, and its failure mode is visible in the same printout.

In [ ]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")      # small; Arctic-Embed-L in production

def take(ds, n, fmt):
    return [fmt(x) for x in itertools.islice(ds, n)]
target = []
target += take(load_dataset("cais/mmlu", "auxiliary_train", split="train", streaming=True), 300,
               lambda x: x["question"] + " " + " ".join(x["choices"]))
target += take(load_dataset("openai/gsm8k", "main", split="train", streaming=True), 300, lambda x: x["question"])
target += take(load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train", streaming=True), 300,
               lambda x: x["question"] + " " + " ".join(x["choices"]["text"]))
print(len(target), "target items")

t0 = time.time()
Q = emb.encode(target, normalize_embeddings=True, batch_size=64)
D = emb.encode([t[:2000] for t in texts], normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print(f"embedded in {time.time()-t0:.0f}s")
sims = Q @ D.T                                              # (items, docs)
ranks = (-sims).argsort(1).argsort(1)                       # rank of each doc under each item (0 = best)
best_rank = ranks.min(0)                                    # BETR's max-aggregation = best position
keep = best_rank <= np.quantile(best_rank, 0.10)             # top 10% of documents by best rank
for d, r, k in zip(docs, best_rank, keep): d["betr_rank"], d["betr_keep"] = int(r), bool(k)
print(f"positives: {keep.sum()} of {len(docs)}")
AUDIT["relevance"] = {"method": "BETR-lite best-rank, all-MiniLM-L6-v2", "targets": len(target), "keep_frac": 0.10}

In [ ]:
order = np.argsort(best_rank)
print("=== MOST on-target (rank 0..2) ===")
for i in order[:3]: print(f"[rank {best_rank[i]}]", texts[i][:260].replace("\n", " "), "\n")
print("=== LEAST on-target ===")
for i in order[-3:]: print(f"[rank {best_rank[i]}]", texts[i][:260].replace("\n", " "), "\n")
# Goodhart check: which target source 'owns' the positives?
owner = sims[:, keep].argmax(0)
src = np.array(["mmlu"]*300 + ["gsm8k"]*300 + ["arc"]*300)
print("positives claimed by:", collections.Counter(src[owner]))

**Read the two printouts.** The top documents are question-shaped and school-subject-flavored — relevance to the suite — and if one source dominates the positives the selected set is drifting toward *that benchmark's format*, which is the Goodhart trap §11.2 warns about. The chapter's remedy: target a broad suite and hold out a second one you never target.

### How much to keep: $F_{\text{opt}}(C) \approx 4\times10^{-5}\,C^{0.25}$

In [ ]:
def f_opt(C): return 4e-5 * C**0.25           # percent of tokens, C in FLOPs (BETR's fit)
for name, N, Dtok in (("1B · 100B", 1e9, 1e11), ("3B · 600B", 3e9, 6e11), ("7B · 2T", 7e9, 2e12), ("70B · 15T", 7e10, 1.5e13)):
    C = 6 * N * Dtok; print(f"{name:>10}: C = {C:.1e} FLOPs → keep top {f_opt(C):.0f}% of tokens")
C = np.logspace(19, 24, 100)
plt.figure(figsize=(6,3)); plt.loglog(C, f_opt(C), color=NAVY, lw=2)
for c, lab in ((1e21, "3%"), (1e22, "10%"), (1e23, "30%")): plt.scatter([c], [f_opt(c)], color=AMBER, zorder=3)
plt.axvline(6*7e9*2e12, color=AMBER, ls="--"); plt.text(6*7e9*2e12*1.1, 3, "7B · 2T", color=AMBER)
plt.xlabel("training FLOPs"); plt.ylabel("% tokens kept"); plt.title("Figure 11.3, recomputed");

> **Exercise 1.1 (hold-out).** Re-run the ranking with only MMLU + ARC as targets and treat GSM8K as the held-out suite. Measure how the GSM8K items' best ranks shift when they are *not* targeted. That shift, scaled up, is BETR's third-place finish on held-out tasks.
>
> **Exercise 1.2 (classifier tier).** Train a fastText or logistic-regression classifier on `betr_keep` (TF-IDF features) and score the documents you did not embed. In production this is how 10M embedded documents become a score for trillions of tokens.

## 2 · Is it good? Calibrating a judge

Chapter 11.3. We treat FineWeb-Edu's `score` column as the **judge** (it is a distilled Llama-3-70B rubric), and build the harness that validates it: the sample-size formula, a stratified labeling sheet for two human annotators, and the κ / confusion computation to run once the sheet is filled. The cells below use *synthetic placeholder labels* so the code runs end-to-end — replace them with the sheet.

In [ ]:
def n_labels(e, p=0.5, z=1.96): return z*z*p*(1-p)/(e*e)
for e in (0.10, 0.05, 0.03, 0.02, 0.01): print(f"±{e:.0%}: {n_labels(e):6.0f} labels")
e = np.linspace(0.01, 0.10, 100)
plt.figure(figsize=(6,3)); plt.plot(e*100, n_labels(e), color=NAVY, lw=2); plt.scatter([3], [n_labels(.03)], color=AMBER, zorder=3)
plt.text(3.3, n_labels(.03)+300, "±3% → 1,070", color=AMBER); plt.xlabel("margin of error (%)"); plt.ylabel("labels"); plt.title("Figure 11.4 left, recomputed");

In [ ]:
# the labeling sheet: stratified by dump (stand-in for source/language), 120 rows for the lab (1,070 in production)
import pandas as pd, random
random.seed(0)
by_dump = collections.defaultdict(list)
for i, d in enumerate(docs): by_dump[d.get("dump", "na")].append(i)
per = max(1, 120 // len(by_dump)); sheet_idx = [i for k in by_dump for i in random.sample(by_dump[k], min(per, len(by_dump[k])))]
sheet = pd.DataFrame({"doc_idx": sheet_idx, "dump": [docs[i].get("dump") for i in sheet_idx],
                      "text": [texts[i][:1500] for i in sheet_idx], "judge_score": [docs[i].get("score") for i in sheet_idx],
                      "annotator_1": "", "annotator_2": ""})      # rubric: 1 = educational (would keep), 0 = not
sheet.to_csv("lab11_labeling_sheet.csv", index=False)
print(sheet[["doc_idx", "dump", "judge_score"]].head(), f"\n→ lab11_labeling_sheet.csv with {len(sheet)} rows; fill annotator_1/2 with 0/1")

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
filled = pathlib.Path("lab11_labeling_sheet_filled.csv")
if filled.exists():
    lab = pd.read_csv(filled)
else:   # PLACEHOLDER labels so the harness runs: noisy functions of the judge — replace with the filled sheet
    rng = np.random.default_rng(0)
    lab = sheet.copy()
    lab["annotator_1"] = ((lab.judge_score + rng.normal(0, .6, len(lab))) >= 3).astype(int)
    lab["annotator_2"] = ((lab.judge_score + rng.normal(0, .6, len(lab))) >= 3).astype(int)
    print("⚠ using synthetic placeholder labels — numbers below are a demo of the harness, not a measurement\n")
h1, h2 = lab.annotator_1.values, lab.annotator_2.values
print(f"human–human κ = {cohen_kappa_score(h1, h2):.2f}   (< 0.6 → fix the rubric before trusting any judge)")
gold = ((h1 + h2) >= 1).astype(int)             # adjudicate disagreements in practice
for thr in (2.5, 3.0, 3.5):
    pred = (lab.judge_score.values >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    print(f"thr {thr}: judge–human κ={cohen_kappa_score(gold, pred):.2f}  precision={tp/max(1,tp+fp):.2f}  recall={tp/max(1,tp+fn):.2f}  false-keep={fp}")
AUDIT["quality"] = {"judge": "fineweb-edu score", "sheet_rows": len(lab), "labels": "placeholder" if not filled.exists() else "human"}

> **Exercise 2.1.** Fill the sheet with a partner (two annotators, blind), save it as `lab11_labeling_sheet_filled.csv`, and re-run. Then read the false-keep rows by hand: do they share a pattern (one dump, one format)? That is a bias to fix, not noise to accept.

## 3 · Does the model already know it? Min-K%, IFD, pass rate

Chapter 11.4 under a small base model (SmolLM2-360M; the chapter's point is that the signal is visible to small models). All three instruments are per-token log-probabilities read in different ways.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"
BASE = "HuggingFaceTB/SmolLM2-360M"
tok = AutoTokenizer.from_pretrained(BASE); tok.pad_token = tok.eos_token
lm = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(dev).eval()

@torch.no_grad()
def token_logprobs(text, max_len=512):
    ids = tok(text, return_tensors="pt", truncation=True, max_length=max_len).input_ids.to(dev)
    lp = torch.log_softmax(lm(ids).logits[0, :-1].float(), -1)
    return lp.gather(1, ids[0, 1:, None])[:, 0].cpu().numpy()
def min_k(lp, k=0.2): return float(np.sort(lp)[: max(1, int(k*len(lp)))].mean())

In [ ]:
# 3a · Min-K% Prob on the candidate documents (N_MK docs) — the membership signal
N_MK = 150 if dev == "cpu" else 1000
mk = []
for t in texts[:N_MK]:
    lp = token_logprobs(t); mk.append((min_k(lp), lp.mean()))
mk = np.array(mk)
plt.figure(figsize=(6,3)); plt.scatter(mk[:,1], mk[:,0], s=10, color=BLUE)
plt.xlabel("mean token log p"); plt.ylabel("Min-20% mean log p"); plt.title("membership lives in the tail (Figure 11.5)")
hi = np.argsort(-mk[:,0])[:2]
print("highest Min-K% (most 'seen' / least surprising):")
for i in hi: print(f"  minK={mk[i,0]:.2f}", texts[i][:200].replace("\n", " "))
AUDIT["membership"] = {"model": BASE, "scored": N_MK, "min_k_percentile_flag": 97}

In [ ]:
# 3b · IFD on UltraChat-200k pairs: ℓ(A|Q) / ℓ(A) — how much the instruction helps
uc = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
pairs = []
for ex in itertools.islice(uc, 60):
    m = ex["messages"]; q, a = m[0]["content"], m[1]["content"]
    if 20 < len(a.split()) < 300: pairs.append((q, a))
def ifd(q, a):
    la = tok(a, add_special_tokens=False).input_ids; n = min(len(la), 300)
    with_q = -token_logprobs(q + "\n" + a, max_len=768)[-n:].mean()
    alone  = -token_logprobs(a, max_len=512).mean()
    return with_q / alone
vals = np.array([ifd(q, a) for q, a in pairs])
plt.figure(figsize=(6,3)); plt.hist(vals, bins=20, color=LIGHT, edgecolor=NAVY); plt.axvline(np.median(vals), color=AMBER, ls="--")
plt.xlabel("IFD = ℓ(A|Q) / ℓ(A)"); plt.title("UltraChat-200k: keep the hard half (IFD above median)")
i_lo, i_hi = vals.argmin(), vals.argmax()
print(f"LOWEST IFD {vals[i_lo]:.2f} (instruction barely matters):\n  Q: {pairs[i_lo][0][:150]!r}\n")
print(f"HIGHEST IFD {vals[i_hi]:.2f} (hard even given the instruction):\n  Q: {pairs[i_hi][0][:150]!r}")
AUDIT["ifd"] = {"dataset": "ultrachat_200k/train_sft", "n": len(vals), "median": float(np.median(vals))}

In [ ]:
# 3c · pass rate on GSM8K train with an instruct model: the bins that teach (0 < p̂ < 1)
inst_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
itok = AutoTokenizer.from_pretrained(inst_name); ilm = AutoModelForCausalLM.from_pretrained(inst_name, torch_dtype=torch.float32).to(dev).eval()
gsm = list(itertools.islice(load_dataset("openai/gsm8k", "main", split="train", streaming=True), 12 if dev == "cpu" else 100))
def gold_answer(a): return a.split("####")[-1].strip().replace(",", "")
def last_number(s):
    m = re.findall(r"-?\d+(?:\.\d+)?", s.replace(",", "")); return m[-1] if m else None
@torch.no_grad()
def sample(q, n):
    msgs = [{"role": "user", "content": q + "\nThink step by step and end with 'The answer is <number>'."}]
    text = itok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = itok(text, return_tensors="pt").input_ids.to(dev)
    out = ilm.generate(ids, do_sample=True, temperature=1.0, top_p=0.95, max_new_tokens=256, num_return_sequences=n, pad_token_id=itok.eos_token_id)
    return [itok.decode(o[ids.shape[1]:], skip_special_tokens=True) for o in out]
K = 4
rates = []
for ex in gsm:
    outs = sample(ex["question"], K)
    rates.append(np.mean([last_number(o) == gold_answer(ex["answer"]) for o in outs]))
rates = np.array(rates)
bins = {"p̂=0 (too hard now)": (rates == 0).sum(), "0<p̂<1 (teaches)": ((rates > 0) & (rates < 1)).sum(), "p̂=1 (known)": (rates == 1).sum()}
print(bins, "\nlearning signal ∝ p̂(1−p̂):", np.round(rates*(1-rates), 2))
AUDIT["pass_rate"] = {"model": inst_name, "k": K, "n_prompts": len(gsm), "bins": {k: int(v) for k, v in bins.items()}}

> **Exercise 3.1.** Raise `K` to 8 and the prompt count to 100 on a GPU. Plot $\hat p(1-\hat p)$ against $\hat p$ and mark where your budget should go. Then re-run after one epoch of SFT on the "teaches" bin only and watch the bins move — that is the curriculum defined by competence (§11.4, and Chapter 25's dynamic sampling).

## 4 · Contamination: the completion test

Chapter 11.5. Give the model the first half of a GSM8K **test** item and ask it to continue, guided ("Complete this item from the GSM8K test split") and unguided; compare ROUGE-L against the true second half with the same test on **fresh controls** of the same kind. A gap well above the controls' noise is contamination. Our controls are GSM8K-style items with the numbers re-sampled — write five of your own for a cleaner control.

In [ ]:
from rouge_score import rouge_scorer
rs = rouge_scorer.RougeScorer(["rougeL"])
test_items = [x["question"] for x in itertools.islice(load_dataset("openai/gsm8k", "main", split="test", streaming=True), 12)]
def perturb(q):   # cheap control: same template, new numbers → the model cannot have seen this string
    return re.sub(r"\d+", lambda m: str(int(m.group()) * 3 + 7), q)
controls = [perturb(q) for q in test_items]
@torch.no_grad()
def complete(prefix, guided):
    hint = "Complete this item from the GSM8K test split.\n" if guided else ""
    ids = tok(hint + prefix, return_tensors="pt").input_ids.to(dev)
    out = lm.generate(ids, do_sample=False, max_new_tokens=48, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
def completion_score(items, guided):
    sc = []
    for q in items:
        w = q.split(); k = len(w) // 2; prefix, truth = " ".join(w[:k]), " ".join(w[k:])
        sc.append(rs.score(truth, complete(prefix, guided))["rougeL"].fmeasure)
    return float(np.mean(sc))
res = {g: (completion_score(test_items, g), completion_score(controls, g)) for g in (False, True)}
for g, (b, c) in res.items(): print(f"{'guided' if g else 'unguided':>9}: GSM8K-test ROUGE-L {b:.3f} | controls {c:.3f} | gap {b-c:+.3f}")
AUDIT["contamination"] = {"test": "completion ROUGE-L, GSM8K test vs perturbed controls", "results": {str(g): v for g, v in res.items()}}

A gap near zero (the usual result for a small open base model on GSM8K) is the honest baseline; try the same cell with a model known to be benchmark-tuned and watch the guided gap open. **Report the gap next to the score** — that is the chapter's rule.

## 5 · The validation record

In [ ]:
print(json.dumps(AUDIT, indent=1, default=str))
pathlib.Path("lab11_audit.json").write_text(json.dumps(AUDIT, indent=1, default=str))

## What you reproduced

* BETR-lite: best-rank targeted selection with a broad suite, the Goodhart check (which source owns the positives), and $F_{\text{opt}}(C)$ for your budget;
* the judge-calibration harness: sample size for ±3% (≈1,070), a stratified labeling sheet, human–human and judge–human κ, and the false-keep cell at three thresholds;
* the three marginal-value instruments under a small base model: Min-K% on documents, IFD on UltraChat pairs, pass-rate bins on GSM8K;
* the completion test for contamination with fresh controls, guided and unguided.

Next: **Lab 27** generates data — structured rephrasing of the Lab 10 slice and Active Reading with a 1.7B generator — and sends it back through this lab's gates.